# Gridded Coverage Validation: SMAP

This notebook validates TAT-C's coverage statistics and their gridding (`aggregate_observations`, `reduce_observations`, and `grid_observations`) against a day of observations by the radiometer of NASA's [Soil Moisture Active Passive](https://smap.jpl.nasa.gov/) (SMAP) mission on 24 October 2026. SMAP's 6 m antenna rotates at about 14.6 revolutions per minute, so its radiometer scans a full circle on the ground around the satellite, about 1,000 km wide, at an incidence angle of 40 deg, viewing each place twice on each pass: once looking forward and once looking aft, about two minutes later. A `ConicalInstrument` that scans the full circle (a half-width of 180 deg) models this geometry, which the AMSR2 and GMI notebooks (sector scanners) do not test.

1. **Looks.** Does `collect_observations` predict each forward and aft look at each cell of SMAP's grid?
2. **Statistics by cell.** Do `aggregate_observations` and `reduce_observations` give each cell's observed number of passes, access, and revisit?
3. **Gridded statistics.** Does `grid_observations` combine them correctly into larger cells?

## Reference Data

SMAP's Level-1C brightness temperatures (`SPL1CTB`, version 6, from the [NSIDC DAAC](https://nsidc.org/data/smap); free [NASA Earthdata Login](https://urs.earthdata.nasa.gov/) account required) are resampled to the 36 km global Equal-Area Scalable Earth grid (EASE-Grid 2.0), one file per half orbit, and record for each cell the time of the forward and aft looks (the weighted mean time of the footprints centered in it). The day's half orbits are downloaded once (keeping the latest version of each), and the look times are cached in a local `data/smap` directory.

The look times are converted to UTC from the seconds since J2000 (noon TT on 1 January 2000). The products' own UTC strings (`cell_tb_time_utc_fore` and `_aft`) add these seconds to noon UTC, and are 69.184 s late (the difference between TT and UTC). The archive lacks a few half orbits late in the day, whose times are excluded from the comparisons.

In [1]:
import re
from datetime import datetime, timedelta, timezone
from pathlib import Path

import h5py
import numpy as np
import pandas as pd

from tatc.constants import timescale

DATA_DIR = Path("data") / "smap"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DAY = datetime(2026, 10, 24, tzinfo=timezone.utc)
CACHE_FILE = DATA_DIR / f"smap_l1c_tb_{DAY:%Y%m%d}.npz"

if not CACHE_FILE.exists():
    import earthaccess

    earthaccess.login()
    granules = earthaccess.search_data(
        short_name="SPL1CTB",
        temporal=(
            f"{DAY - timedelta(hours=2):%Y-%m-%dT%H:%M:%S}",
            f"{DAY + timedelta(hours=26):%Y-%m-%dT%H:%M:%S}",
        ),
    )
    # the latest version of each half orbit (orbit number and direction)
    latest = {}
    for g in sorted(granules, key=lambda g: g["umm"]["GranuleUR"]):
        name = g["umm"]["GranuleUR"]
        latest[re.search(r"_TB_(\d+_[AD])_", name).group(1)] = g
    files = earthaccess.download(list(latest.values()), str(DATA_DIR / "raw"))
    parts = []
    for k, path in enumerate(sorted(files)):
        with h5py.File(path) as f:
            cells = f["Global_Projection"]
            part = {
                "row": cells["cell_row"][:],
                "col": cells["cell_column"][:],
                "incidence": cells["cell_boresight_incidence_fore"][:],
            }
            for look in ("fore", "aft"):
                # seconds since J2000 (noon TT on 1 January 2000), converted to UTC
                seconds = cells[f"cell_tb_time_seconds_{look}"][:]
                valid = seconds > -9999
                utc = pd.to_datetime(
                    timescale.tt_jd(2451545.0 + seconds[valid] / 86400).utc_datetime(),
                    utc=True,
                )
                part[look] = np.full(len(seconds), np.nan)
                part[look][valid] = (utc - pd.Timestamp(DAY)).total_seconds()
            part["half_orbit"] = np.full(len(part["row"]), k)
            parts.append(part)
    np.savez_compressed(
        CACHE_FILE, **{key: np.concatenate([p[key] for p in parts]) for key in parts[0]}
    )
    for path in files:
        Path(
            path
        ).unlink()  # keep only the cached looks (each half orbit is about 5 MB)

smap = dict(np.load(CACHE_FILE))
# looks (forward and aft) within the day, by cell
looks = pd.concat(
    [
        pd.DataFrame(
            {
                "row": smap["row"],
                "col": smap["col"],
                "half_orbit": smap["half_orbit"],
                "seconds": smap[look],
                "look": look,
            }
        )
        for look in ("fore", "aft")
    ]
).dropna(subset=["seconds"])
looks = looks[(looks.seconds >= 0) & (looks.seconds < 86400)]
# data gaps: periods of more than a minute without a look
covered = looks.groupby("half_orbit").seconds.agg(["min", "max"]).sort_values("min")
GAPS = [
    (a, b)
    for a, b in zip(
        covered["max"].cummax().to_numpy()[:-1], covered["min"].to_numpy()[1:]
    )
    if b - a > 60
]
if covered["min"].iloc[0] > 60:
    GAPS.insert(0, (0, covered["min"].iloc[0]))
if covered["max"].max() < 86400 - 60:
    GAPS.append((covered["max"].max(), 86400))
print(
    f"{smap['half_orbit'].max() + 1} half orbits, {len(looks)} looks within the day of {looks.groupby(['row', 'col']).ngroups} cells"
)
print(
    "data gaps (UTC): "
    + ", ".join(
        f"{DAY + timedelta(seconds=a):%H:%M}-{DAY + timedelta(seconds=b):%H:%M}"
        for a, b in GAPS
    )
)

32 half orbits, 890452 looks within the day of 273726 cells
data gaps (UTC): 21:01-22:35, 23:29-00:00


## Setup

SMAP is modeled as a `ConicalInstrument` scanning the full circle, with the cone angle of the median incidence angle of its looks at its mean altitude (on a spherical Earth), with an along-track field of view of 0.01 deg (so that each observation period spans the forward to the aft crossing of the circle), on an orbit from a published two-line element set (TLE) with an epoch about a day and a half after the analysis day. The points are the centers of every other cell of every other row of the EASE-Grid 2.0 (about 72 km apart), between about 81 deg south and 84 deg north. `collect_observations` reports each pass over a point as two observations, at the forward and aft crossings of the circle (`epoch`), which share the period during which the point lies inside the circle; its run of about half a minute is cached in `data/smap`.

In [2]:
import pickle
import time

from pyproj import Transformer
from shapely.geometry import Point

from tatc.analysis import (
    aggregate_observations,
    collect_observations,
    grid_observations,
    reduce_observations,
)
from tatc.schemas import ConicalInstrument, GeneralPerturbationsOrbit, Satellite

TLE = [
    "1 40376U 15003A   26298.67804942  .00000247  00000+0  56844-4 0  9999",
    "2 40376  98.1310 303.9506 0001331  85.8091 274.3263 14.63397245626701",
]
EARTH_RADIUS = 6371.0088e3  # m, mean
# EASE-Grid 2.0 global, 36 km: cell size (m) and the projected coordinates of the upper left corner
CELL, X0, Y0 = 36032.220840584, -17367530.44516138, 7314540.835948884

orbit = GeneralPerturbationsOrbit.from_tle(TLE)
incidence = float(np.nanmedian(smap["incidence"]))
# the cone angle (from nadir) of a boresight at the median incidence angle, at the mean altitude (spherical Earth)
cone = float(
    np.degrees(
        np.arcsin(
            EARTH_RADIUS
            * np.sin(np.radians(incidence))
            / (EARTH_RADIUS + orbit.get_mean_altitude())
        )
    )
)
smap_sat = Satellite(
    name="SMAP",
    orbit=orbit,
    instruments=[
        ConicalInstrument(
            name="radiometer",
            cone_angle=cone,
            scan_center_azimuth=0,
            scan_half_width=180,
            along_track_field_of_view=0.01,
        )
    ],
)
# points at the centers of every other cell of every other row
rows, cols = np.meshgrid(np.arange(0, 406, 2), np.arange(0, 964, 2), indexing="ij")
lon, lat = Transformer.from_crs("EPSG:6933", "EPSG:4326", always_xy=True).transform(
    X0 + (cols.ravel() + 0.5) * CELL, Y0 - (rows.ravel() + 0.5) * CELL
)
points = pd.DataFrame(
    {"row": rows.ravel(), "col": cols.ravel(), "lon": lon, "lat": lat}
)
point_index = pd.Series(
    np.arange(len(points)), index=pd.MultiIndex.from_arrays([points.row, points.col])
)
looks["point"] = point_index.reindex(
    pd.MultiIndex.from_arrays([looks.row, looks.col])
).to_numpy()
looks = looks.dropna(subset=["point"]).astype({"point": int})

cache = DATA_DIR / f"observations_{DAY:%Y%m%d}.pkl"
if not cache.exists():
    tic = time.perf_counter()
    found = collect_observations(
        [Point(x, y) for x, y in zip(lon, lat)], smap_sat, DAY, DAY + timedelta(days=1)
    )
    print(
        f"collect_observations: {len(found)} observations of {len(points)} points in {time.perf_counter() - tic:.0f} s"
    )
    found.to_pickle(cache)
observations = pd.read_pickle(cache)
hash_index = pd.Series(
    np.arange(len(points)), index=[Point(x, y).wkb for x, y in zip(lon, lat)]
)
observations["point"] = observations.geometry.apply(
    lambda g: hash_index[Point(g.x, g.y).wkb]
).to_numpy()
observations["seconds"] = (observations.epoch - pd.Timestamp(DAY)).dt.total_seconds()


def in_gap(seconds, margin=180):
    """Whether times (s) fall within a margin (s) of a data gap (a pass's looks span about two minutes)."""
    seconds = np.asarray(seconds)
    return np.any(
        [(seconds > a - margin) & (seconds < b + margin) for a, b in GAPS]
        + [np.zeros(seconds.shape, bool)],
        axis=0,
    )


print(
    f"median incidence angle {incidence:.2f} deg, mean altitude {orbit.get_mean_altitude() / 1e3:.1f} km: cone angle {cone:.3f} deg"
)
print(
    f"{len(points)} points, latitudes {lat.min():.1f} to {lat.max():.1f} deg; {len(observations)} observations"
)

collect_observations: 230357 observations of 97846 points in 35 s


median incidence angle 39.97 deg, mean altitude 689.4 km: cone angle 35.431 deg
97846 points, latitudes -81.5 to 83.6 deg; 230357 observations


## Test 1: Looks

Each observed look (forward or aft) at a point is paired with the nearest predicted observation of the same point within 60 s, outside the data gaps.

In [3]:
def pair(left, right, tolerance=60):
    """Pairs each row of left with the nearest row of right at the same point within tolerance (s) of time."""
    return pd.merge_asof(
        left.sort_values("seconds"),
        right.rename(columns={"seconds": "paired"}).sort_values("paired"),
        left_on="seconds",
        right_on="paired",
        by="point",
        direction="nearest",
        tolerance=tolerance,
    )


observed = looks[~in_gap(looks.seconds)]
predicted = observations[~in_gap(observations.seconds)]
forward = pair(observed[["point", "seconds", "look"]], predicted[["point", "seconds"]])
backward = pair(predicted[["point", "seconds"]], observed[["point", "seconds"]])
offset = (forward.paired - forward.seconds).dropna()
pd.DataFrame(
    {
        look: {
            "observed looks": int((forward.look == look).sum()),
            "predicted (%)": 100 * forward.paired[forward.look == look].notna().mean(),
            "time offset, median (s)": offset[forward.look == look].median(),
            "95% |offset| (s)": offset[forward.look == look].abs().quantile(0.95),
        }
        for look in ("fore", "aft")
    }
    | {
        "all predicted looks": {
            "observed looks": len(predicted),
            "predicted (%)": 100 * backward.paired.notna().mean(),
        }
    }
).T.round(2)

,observed looks,predicted (%),"time offset, median (s)",95% |offset| (s)
fore,110535.0,95.28,-0.19,1.60
aft,110559.0,95.28,0.11,1.54
all predicted looks,208936.0,96.63,NaN,NaN


A full-circle `ConicalInstrument` reproduces SMAP's two looks per pass: `collect_observations` predicts 95.3% of the forward and the aft looks alike, within 0.2 s (median) and 1.6 s (95%) of their times, and 96.6% of its predicted looks were observed. Every look not predicted belongs to a pass over a point that TAT-C does not predict at all, not to a misplaced one: these points lie just outside TAT-C's swath, where SMAP records a look at a cell whenever a footprint is centered anywhere within it (up to half a cell, 18 km, from its center), and are more common at high latitudes (a median of 64 deg, against 47 deg for all looks), where the edges of successive swaths cross more cells.

## Test 2: Statistics by Cell

`aggregate_observations` merges each point's overlapping observations into periods (here, a pass's forward and aft observations, which share a period), and `reduce_observations` gives each point's number of samples (passes), mean access (the duration of the periods), and mean revisit (the time between them). The observed statistics are computed the same way from each pass's looks: a period from the first look to the last (forward to aft). Observations within the data gaps are excluded from both.

In [4]:
def statistics(periods):
    """Samples, mean access (s), and mean revisit (s) of each point, from its observation periods (point, start,
    end, in s), merged where they overlap (as by aggregate_observations)."""
    periods = periods.sort_values(["point", "start"])
    first = periods.point.ne(periods.point.shift())
    reach = periods.groupby("point").end.cummax()
    periods = periods.assign(
        group=(first | (periods.start > reach.groupby(periods.point).shift())).cumsum()
    )
    merged = periods.groupby("group").agg(
        point=("point", "first"), start=("start", "min"), end=("end", "max")
    )
    merged["access"] = merged.end - merged.start
    merged["revisit"] = merged.start - merged.end.groupby(merged.point).shift()
    return merged.groupby("point").agg(
        samples=("start", "size"),
        access=("access", "mean"),
        revisit=("revisit", "mean"),
    )


# observed: each half orbit's pass over a point, from its first to its last look
passes = (
    looks[~in_gap(looks.seconds)]
    .groupby(["point", "half_orbit"])
    .seconds.agg(start="min", end="max")
    .reset_index()
)
observed = statistics(passes)
# predicted: aggregate_observations and reduce_observations, outside the data gaps
tic = time.perf_counter()
aggregated = aggregate_observations(
    observations[~in_gap(observations.seconds)].drop(columns=["point", "seconds"])
)
reduced = reduce_observations(aggregated)
elapsed = time.perf_counter() - tic
reduced["point"] = reduced.geometry.apply(
    lambda g: hash_index[Point(g.x, g.y).wkb]
).to_numpy()
predicted = reduced.set_index("point")[["samples", "access", "revisit"]].assign(
    access=lambda f: f.access.dt.total_seconds(),
    revisit=lambda f: f.revisit.dt.total_seconds(),
)
both = observed.add_suffix("_observed").join(
    predicted.add_suffix("_predicted"), how="outer"
)
both[["samples_observed", "samples_predicted"]] = both[
    ["samples_observed", "samples_predicted"]
].fillna(0)
difference = both.samples_predicted - both.samples_observed
same = (difference == 0) & both.revisit_observed.notna()
print(
    f"aggregate_observations and reduce_observations: {len(observations)} observations in {elapsed:.1f} s"
)
pd.Series(
    {
        "points observed": int((both.samples_observed > 0).sum()),
        "predicted": int((both.samples_predicted > 0).sum()),
        "same number of passes (%)": 100 * (difference == 0).mean(),
        "one more predicted (%)": 100 * (difference == 1).mean(),
        "one fewer predicted (%)": 100 * (difference == -1).mean(),
        "other (%)": 100 * (difference.abs() > 1).mean(),
        "access difference, median (s)": (
            both.access_predicted - both.access_observed
        ).median(),
        "95% |access difference| (s)": (both.access_predicted - both.access_observed)
        .abs()
        .quantile(0.95),
        "revisit difference (same passes), median (s)": (
            both.revisit_predicted - both.revisit_observed
        )[same].median(),
        "95% |revisit difference| (s)": (
            both.revisit_predicted - both.revisit_observed
        )[same]
        .abs()
        .quantile(0.95),
    }
).round(2)

aggregate_observations and reduce_observations: 230357 observations in 4.1 s


points observed                                 68435.00
predicted                                       68485.00
same number of passes (%)                          89.82
one more predicted (%)                              3.86
one fewer predicted (%)                             5.63
other (%)                                           0.70
access difference, median (s)                       7.50
95% |access difference| (s)                        29.12
revisit difference (same passes), median (s)       -7.78
95% |revisit difference| (s)                       16.13
dtype: float64

`aggregate_observations` and `reduce_observations` give the same number of passes for 90% of the points observed or predicted (68,435 and 68,485), one more for 3.9%, and one fewer for 5.6%, at the edges of the swaths as in Test 1, in about 4 s for the 230,000 observations of the day. Their access periods are 7.5 s longer (median) than the observed ones, from the forward to the aft look, and their revisits correspondingly shorter: by default, a `ConicalInstrument`'s field of regard, whose access defines each observation's period, extends 1 deg beyond its cone (about 3.4 s on either side of the circle); the epochs themselves match (Test 1).

## Test 3: Gridded Statistics

`grid_observations` combines the statistics of the points within each cell of a coarser grid, here TAT-C's cells about 1,000 km wide (`generate_cells_uniform_spacing`): the total number of samples, the sample-weighted mean access, and the sample-weighted harmonic mean revisit (the mean of the revisit rates). The observed statistics of Test 2 are gridded the same way into the same cells.

In [5]:
import geopandas as gpd

from tatc.generation import generate_cells_uniform_spacing

cells = generate_cells_uniform_spacing(1000e3)
tic = time.perf_counter()
gridded = grid_observations(reduced, cells)
elapsed = time.perf_counter() - tic
# the same statistics gridded from the observed points: total samples, sample-weighted mean access, and
# sample-weighted harmonic mean revisit
located = gpd.GeoDataFrame(
    observed.join(points[["lon", "lat"]]),
    geometry=gpd.points_from_xy(points.lon, points.lat)[observed.index],
    crs="EPSG:4326",
).sjoin(cells[["cell_id", "geometry"]], predicate="within")
located["revisit_samples"] = located.samples.where(located.revisit.notna(), 0)
located["rate"] = (located.samples / located.revisit).fillna(0)
located["access_x_samples"] = located.access * located.samples
truth = located.groupby("cell_id")[
    ["samples", "access_x_samples", "revisit_samples", "rate"]
].sum()
truth["access"] = truth.access_x_samples / truth.samples
truth["revisit"] = (truth.revisit_samples / truth.rate).where(truth.rate > 0)
compared = (
    gridded.set_index("cell_id")[["samples", "access", "revisit"]]
    .assign(
        access=lambda f: pd.to_timedelta(f.access).dt.total_seconds(),
        revisit=lambda f: pd.to_timedelta(f.revisit).dt.total_seconds(),
    )
    .add_suffix("_predicted")
    .join(truth[["samples", "access", "revisit"]].add_suffix("_observed"), how="inner")
)
ratio = compared.samples_predicted / compared.samples_observed
print(
    f"{len(cells)} cells, {len(gridded)} with observations, gridded in {elapsed:.1f} s; {len(compared)} compared"
)
pd.DataFrame(
    {
        "samples (predicted / observed)": ratio.quantile([0.05, 0.5, 0.95]),
        "access (predicted / observed)": (
            compared.access_predicted / compared.access_observed
        ).quantile([0.05, 0.5, 0.95]),
        "revisit (predicted / observed)": (
            compared.revisit_predicted / compared.revisit_observed
        ).quantile([0.05, 0.5, 0.95]),
    }
).T.rename(columns=lambda q: f"{q:.0%}").round(3)

800 cells, 780 with observations, gridded in 1.1 s; 780 compared


,5%,50%,95%
samples (predicted / observed),0.841,0.976,1.086
access (predicted / observed),1.063,1.120,1.500
revisit (predicted / observed),0.947,1.000,1.098


Gridded into 800 cells about 1,000 km wide, the statistics agree with the observed ones gridded the same way: the median cell has 97.6% of the observed samples (84 to 109% for 90% of the cells, from the swath edges of Test 2), the same revisit (a ratio of 1.00, and within 5 to 10% for 90% of the cells), and an access longer by the field of regard's margin (12%, median), in about a second. In cells whose points each have a single sample, the revisit is undefined.

## Summary

| Test | Result |
| --- | --- |
| 1. Looks | 95.3% of SMAP's forward and aft looks predicted, within 0.2 s (median) and 1.6 s (95%); 96.6% of predicted looks observed; the others are whole passes at the swath edges |
| 2. Statistics by cell | Same number of passes for 90% of the points; access 7.5 s longer (median) by the field of regard's default margin |
| 3. Gridded statistics | Samples within 2.4% (median) and revisit equal (median) in 1,000 km cells; access longer by the margin |

**TAT-C's coverage statistics are validated against SMAP.** A full-circle `ConicalInstrument` predicts SMAP's forward and aft looks to a fraction of a second, and `aggregate_observations`, `reduce_observations`, and `grid_observations` reproduce its passes, revisits, and their gridded means, for a day of observations of nearly 100,000 points in seconds.

**Findings.** The validation exposed two problems, both corrected: `grid_observations` failed (with an infinite revisit) for a cell whose points each had a single sample, and included those points' samples in the weights of the cell's mean revisit; and `aggregate_observations` took about five minutes for a day of observations of 100,000 points, which it now computes for all points together in a few seconds, with the same results. The reference data needed care too: the products' UTC strings are 69.184 s late (they add seconds since J2000, a TT epoch, to noon UTC), and the archive lacked a few half orbits.

**Next steps:**

1. *Polar grids.* SMAP's products also include polar EASE grids, which would extend the comparison over the poles.
2. *Longer periods.* Over SMAP's 8-day repeat cycle, the revisit statistics would compare the variation between days.

**Limitations.** This study covers one day. The points are the centers of every other cell, where SMAP observes a cell from any footprint centered within it, so passes differ at the swath edges, and the cone angle comes from a spherical Earth.